In [12]:
# ============================================================
# TWO-STAGE HYBRID CLASSIFIER — FINAL VERSION (STRATEGY C)
# ============================================================

import pandas as pd
import numpy as np

from collections import Counter, defaultdict
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# ============================================================
# CONFIG
# ============================================================
DEV_PATH  = "../data/raw/development.csv"
EVAL_PATH = "../data/raw/evaluation.csv"
SUB_PATH  = "submission_v2.csv"

MIN_RULE_SUPPORT = 30
MIN_RULE_PURITY  = 0.95
RULE_PRIORITY    = "best_purity_then_freq"
C_VALUE          = 1.5

# ============================================================
# LOAD DATA
# ============================================================
df_dev  = pd.read_csv(DEV_PATH)
df_eval = pd.read_csv(EVAL_PATH)

# ============================================================
# TIMESTAMP PARSE
# ============================================================
for df in [df_dev, df_eval]:
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

# DROP TIMESTAMP NaN **ONLY ON DEV**

print("DEV samples:", len(df_dev))
print("EVAL samples:", len(df_eval))

# ============================================================
# BASIC FIXES
# ============================================================
for df in [df_dev, df_eval]:
    df["article"] = df["article"].fillna("").astype(str)
    df["title"]   = df["title"].fillna("").astype(str)
    df["source"]  = df["source"].fillna("").astype(str)

# ============================================================
# TEXT
# ============================================================
def build_text(df):
    return (df["title"] + " " + df["article"]).str.lower()

df_dev["text"]  = build_text(df_dev)
df_eval["text"] = build_text(df_eval)

# ============================================================
# NUMERIC FEATURES
# ============================================================
def add_numeric(df):
    df["n_tokens"]    = df["article"].str.split().str.len()
    df["title_len"]   = df["title"].str.len()
    df["article_len"] = df["article"].str.len()
    df["title_ratio"] = df["title_len"] / (df["article_len"] + 1)
    df["year"]  = df["timestamp"].dt.year
    df["month"] = df["timestamp"].dt.month
    df["dow"]   = df["timestamp"].dt.dayofweek
    return df

df_dev  = add_numeric(df_dev)
df_eval = add_numeric(df_eval)

NUM_COLS = [
    "n_tokens", "title_len", "article_len",
    "title_ratio", "year", "month", "dow"
]

# IMPORTANT: fill NaN numerics (EVAL SAFE)
for df in [df_dev, df_eval]:
    df[NUM_COLS] = df[NUM_COLS].fillna(0)

# ============================================================
# FEATURES
# ============================================================
FEATURES = [
    "source", "text",
    "n_tokens", "title_len", "article_len",
    "title_ratio", "year", "month", "dow"
]

X_dev  = df_dev[FEATURES]
y_dev  = df_dev["label"].astype(int)
X_eval = df_eval[FEATURES]

# ============================================================
# RULE MINING
# ============================================================
def tokenize_for_rules(text):
    return text.split()

def mine_pure_rules(texts, labels):
    counts = defaultdict(lambda: Counter())

    for txt, y in zip(texts, labels):
        for tok in set(tokenize_for_rules(txt)):
            counts[tok][y] += 1

    rule_token_to_class = {}
    rule_meta = {}

    for tok, c in counts.items():
        total = sum(c.values())
        if total < MIN_RULE_SUPPORT:
            continue

        best_class, best_freq = c.most_common(1)[0]
        purity = best_freq / total

        if purity >= MIN_RULE_PURITY:
            rule_token_to_class[tok] = best_class
            rule_meta[tok] = (purity, total)

    return rule_token_to_class, rule_meta

def apply_rules(texts, rule_token_to_class, rule_meta):
    rule_pred = np.full(len(texts), -1, dtype=int)

    for i, txt in enumerate(texts):
        toks = set(tokenize_for_rules(txt))
        hits = [t for t in toks if t in rule_token_to_class]
        if not hits:
            continue

        if RULE_PRIORITY == "best_purity_then_freq":
            hits.sort(key=lambda t: (rule_meta[t][0], rule_meta[t][1]), reverse=True)
        else:
            hits.sort(key=lambda t: rule_meta[t][1], reverse=True)

        rule_pred[i] = rule_token_to_class[hits[0]]

    return rule_pred

# ============================================================
# MODEL
# ============================================================
def make_model():
    pre = ColumnTransformer(
        transformers=[
            ("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),

            ("w_tfidf", TfidfVectorizer(
                analyzer="word",
                ngram_range=(1,2),
                min_df=3,
                max_df=0.9,
                sublinear_tf=True,
                max_features=250_000
            ), "text"),

            ("c_tfidf", TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(3,5),
                min_df=3,
                max_df=0.9,
                sublinear_tf=True,
                max_features=300_000
            ), "text"),

            ("num", StandardScaler(), NUM_COLS)
        ],
        remainder="drop",
        n_jobs=-1
    )

    clf = LogisticRegression(
        C=C_VALUE,
        class_weight="balanced",
        max_iter=2000,
        n_jobs=-1
    )

    return Pipeline([
        ("pre", pre),
        ("clf", clf)
    ])

# ============================================================
# TRAIN ON FULL DEV
# ============================================================
print("\nTraining model on full development...")
model = make_model()
model.fit(X_dev, y_dev)

print("Mining rules on full development...")
rule_token_to_class, rule_meta = mine_pure_rules(df_dev["text"], y_dev)
print("Total rules:", len(rule_token_to_class))

# ============================================================
# PREDICT ON EVAL
# ============================================================
print("Predicting on evaluation...")
model_pred = model.predict(X_eval)
rule_pred  = apply_rules(df_eval["text"], rule_token_to_class, rule_meta)

final_pred = model_pred.copy()
mask = rule_pred != -1
final_pred[mask] = rule_pred[mask]

print(f"Rule coverage on eval: {mask.mean():.3f}")

# ============================================================
# SUBMISSION
# ============================================================
submission = pd.DataFrame({
    "Id": df_eval["Id"],
    "label": final_pred
})

submission.to_csv(SUB_PATH, index=False)
print("Submission saved to:", SUB_PATH)



DEV samples: 79997
EVAL samples: 20000

Training model on full development...
Mining rules on full development...
Total rules: 103
Predicting on evaluation...
Rule coverage on eval: 0.055
Submission saved to: submission_v2.csv


In [6]:
df_eval["Id"].isna().sum()


np.int64(0)

In [5]:
EVAL_PATH = "../data/raw/evaluation.csv"
df_eval = pd.read_csv(EVAL_PATH)

In [4]:

import pandas as pd
import numpy as np

from collections import Counter, defaultdict
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [7]:
(df_eval["Id"].astype(str).str.strip() == "").sum()

np.int64(0)

In [8]:
df_eval["Id"].dtype
df_eval["Id"].head(10)

0    0
1    1
2    2
3    3
4    4
5    5
6    6
7    7
8    8
9    9
Name: Id, dtype: int64

In [9]:
df_eval["Id"].nunique(), len(df_eval)

(20000, 20000)

In [10]:
df_eval["Id"].dtype

dtype('int64')

In [11]:
df_eval["Id"] = df_eval["Id"].astype(int)

In [13]:
sub = pd.read_csv("submission_v2.csv")

In [15]:
submission["Id"].isna().sum()

np.int64(0)

In [16]:
submission.to_csv("submission_v4.csv", index=False)
print("Submission saved to:", SUB_PATH)


Submission saved to: submission_v2.csv


In [17]:
print(submission.head())
print(submission.dtypes)
print("NaN count:\n", submission.isna().sum())
print("Inf count:\n", np.isinf(submission.select_dtypes(include=[np.number])).sum())
print("Unique labels:", sorted(submission["label"].unique())[:20])
print("Label dtype:", submission["label"].dtype)
print("Id dtype:", submission["Id"].dtype)

   Id  label
0   0      5
1   1      2
2   2      5
3   3      0
4   4      5
Id       int64
label    int64
dtype: object
NaN count:
 Id       0
label    0
dtype: int64
Inf count:
 Id       0
label    0
dtype: int64
Unique labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
Label dtype: int64
Id dtype: int64


In [18]:
import joblib
import json

# salva il modello sklearn
joblib.dump(model, "model_strategy_c.joblib")

# salva le rules
joblib.dump(rule_token_to_class, "rules_token_to_class.joblib")
joblib.dump(rule_meta, "rules_meta.joblib")

print("Model and rules saved.")


Model and rules saved.


In [19]:
import joblib
import numpy as np
import pandas as pd

# ricarica
model = joblib.load("model_strategy_c.joblib")
rule_token_to_class = joblib.load("rules_token_to_class.joblib")
rule_meta = joblib.load("rules_meta.joblib")

# predizione
model_pred = model.predict(X_eval)

rule_pred = apply_rules(
    df_eval["text"],
    rule_token_to_class,
    rule_meta
)

final_pred = model_pred.copy()
final_pred[rule_pred != -1] = rule_pred[rule_pred != -1]

# QUI la cosa cruciale:
final_pred = np.asarray(final_pred, dtype=int)


In [22]:
submission = pd.DataFrame({
    "Id": df_eval["Id"].values,          # ARRAY
    "Predicted": final_pred                  # ARRAY
})

# check obbligatori
assert submission.isna().sum().sum() == 0
assert len(submission) == len(df_eval)

submission.to_csv("submission_debug2.csv", index=False)


In [2]:
# ============================
# BASELINE MODEL + ERROR ANALYSIS (DEVELOPMENT ONLY)
# ============================

# ---------- IMPORTS ----------
import re
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix

# ---------- PATH ----------
DEV_PATH = "../data/raw/development.csv"

# ---------- TOKENIZER (regex-based, stable for HTML / URL) ----------
def tokenize_for_rules(text):
	return re.findall(r"[a-z0-9_:/\.]+", text.lower())

# ---------- LOAD DATA ----------
df = pd.read_csv(DEV_PATH)

df["article"] = df["article"].fillna("").astype(str)
df["title"]   = df["title"].fillna("").astype(str)
df["source"]  = df["source"].fillna("").astype(str)
df["label"]   = df["label"].astype(int)

# ---------- TEXT BUILD ----------
def build_model_text(df):
	return (df["title"] + " " + df["article"]).str.lower()

df["text"] = build_model_text(df)

# ---------- NUMERIC FEATURES ----------
def add_numeric(df):
	df["n_tokens"]    = df["article"].str.split().str.len()
	df["title_len"]   = df["title"].str.len()
	df["article_len"] = df["article"].str.len()
	df["title_ratio"] = df["title_len"] / (df["article_len"] + 1)
	return df

df = add_numeric(df)

NUM_COLS = ["n_tokens", "title_len", "article_len", "title_ratio"]

df[NUM_COLS] = df[NUM_COLS].replace([np.inf, -np.inf], 0).fillna(0)

# ---------- MODEL ----------
C_VALUE = 1.5

def make_model():
	pre = ColumnTransformer(
		transformers=[
			("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),
			("w_tfidf", TfidfVectorizer(
				analyzer="word",
				tokenizer=tokenize_for_rules,
				ngram_range=(1, 2),
				min_df=3,
				max_df=0.9,
				sublinear_tf=True,
				max_features=250_000
			), "text"),
			("c_tfidf", TfidfVectorizer(
				analyzer="char_wb",
				ngram_range=(3, 5),
				min_df=3,
				max_df=0.9,
				sublinear_tf=True,
				max_features=300_000
			), "text"),
			("num", StandardScaler(), NUM_COLS),
		],
		remainder="drop",
		n_jobs=-1
	)

	clf = LogisticRegression(
		C=C_VALUE,
		class_weight="balanced",
		max_iter=2000,
		n_jobs=-1
	)

	return Pipeline([
		("pre", pre),
		("clf", clf),
	])

# ---------- CROSS-VALIDATION + ERROR COLLECTION ----------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

errors = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(df, df["label"]), 1):
	print(f"\n===== FOLD {fold} =====")

	df_tr = df.iloc[tr_idx]
	df_va = df.iloc[va_idx]

	model = make_model()
	model.fit(df_tr, df_tr["label"])

	y_pred = model.predict(df_va)

	print(classification_report(df_va["label"], y_pred, digits=3))
	print("Confusion Matrix:\n", confusion_matrix(df_va["label"], y_pred))

	fold_errors = df_va.copy()
	fold_errors["y_true"] = df_va["label"].values
	fold_errors["y_pred"] = y_pred
	fold_errors["fold"]   = fold

	fold_errors = fold_errors[fold_errors["y_true"] != fold_errors["y_pred"]]
	errors.append(fold_errors)

# ---------- FINAL ERROR DATAFRAME ----------
errors_df = pd.concat(errors, axis=0).reset_index(drop=True)

print("\nTOTAL MISCLASSIFIED SAMPLES:", len(errors_df))
print(errors_df[["fold", "y_true", "y_pred"]].head())

# ---------- SAVE ----------
errors_df.to_csv("baseline_misclassified_development.csv", index=False)

print("\nSaved: baseline_misclassified_development.csv")




===== FOLD 1 =====
              precision    recall  f1-score   support

           0      0.792     0.705     0.746      4708
           1      0.755     0.820     0.786      2117
           2      0.822     0.827     0.825      2232
           3      0.564     0.568     0.566      1996
           4      0.829     0.913     0.869      1715
           5      0.538     0.521     0.529      2611
           6      0.587     0.795     0.675       621

    accuracy                          0.716     16000
   macro avg      0.698     0.736     0.714     16000
weighted avg      0.718     0.716     0.715     16000

Confusion Matrix:
 [[3319  154  107  294   41  700   93]
 [  78 1735  108   69   15   72   40]
 [  66  127 1846   77    8   52   56]
 [ 186  115  109 1134  124  260   68]
 [  12   13    2   69 1566   47    6]
 [ 499  141   61  337  128 1360   85]
 [  30   12   12   32    6   35  494]]

===== FOLD 2 =====
              precision    recall  f1-score   support

           0      0.79